In [1]:
###############
# Directories #
############### 
directory = 'C:\\Users\\20234503\\Dropbox\\PhD Project\\Research\\Weather and Strikes\\Project\\' 
##########
#Packages#
##########
import os
import requests
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

import xarray as xr
import pandas as pd
import geopandas as gpd
from shapely.geometry import box
from pyproj import CRS, Transformer

from shapely import wkt

from shapely.geometry import Point
from sklearn.neighbors import BallTree
import numpy as np
import matplotlib.pyplot as plt

from shapely.ops import unary_union

# --- keep this near your imports ---
import types
import inspect
import sys

def clear_env(keep=None):
    """
    Delete user-defined globals while keeping:
      - all imported modules (e.g. pd, np, gpd, plt, string, etc.)
      - functions/classes imported via `from ... import ...` (e.g. unary_union, Counter, Patch)
      - anything explicitly listed in `keep`
      - underscore-prefixed names and this function itself
    """
    if keep is None:
        keep = []
    keep = set(keep) | {"clear_env"}

    g = globals()
    for name, val in list(g.items()):
        # never touch dunders, explicit keepers, or this function
        if name.startswith("_") or name in keep:
            continue

        # keep modules (e.g., pd, np, gpd, plt, string, re, json, sys, types, etc.)
        if isinstance(val, types.ModuleType):
            continue

        # keep imported functions/classes (e.g., unary_union, Counter, Patch, product)
        # (callables or types) whose defining module is not __main__
        if (callable(val) or isinstance(val, type)):
            mod = inspect.getmodule(val)
            if mod is not None and mod.__name__ != "__main__":
                continue

        # otherwise, remove it (likely user-created data/vars/objects)
        try:
            del g[name]
        except Exception:
            pass

# Checked if code is correct 13/11/2025



In [ ]:
############################
# Downloading Weather Data #
############################
# === CONFIG ===
base_url = "https://www.ncei.noaa.gov/pub/data/daily-grids/v1-0-0/grids"
output_folder = directory + "Data\\Raw\\Weather\\"
os.makedirs(output_folder, exist_ok=True)

start_year = 1991 #1991
end_year = 2025 # Adjust the years as needed
# Note: The dataset is available from 1951 onwards, so adjust the start_year accordingly
max_workers = 5  # number of parallel downloads

# === Generate the full list of URLs and paths ===
download_list = []

for year in range(start_year, end_year + 1):
    for month in range(1, 13):
        yyyy = f"{year:04d}"
        yyyymm = f"{year:04d}{month:02d}"
        file_name = f"ncdd-{yyyymm}-grd-scaled.nc"
        url = f"{base_url}/{yyyy}/{file_name}"
        output_path = os.path.join(output_folder, file_name)
        download_list.append((url, output_path, file_name))

# === Download function ===
def download_file(url, output_path, file_name):
    if os.path.exists(output_path):
        return f"✅ Already exists: {file_name}"

    try:
        response = requests.get(url, stream=True, timeout=60)
        if response.status_code == 404:
            return f"⚠️ Not found: {file_name}"

        response.raise_for_status()

        with open(output_path, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)

        return f"✅ Downloaded: {file_name}"

    except requests.exceptions.RequestException as e:
        return f"❌ Failed: {file_name} ({e})"

# === Parallel download with ThreadPoolExecutor ===
with ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures = [executor.submit(download_file, url, path, name) for url, path, name in download_list]

    for future in tqdm(as_completed(futures), total=len(futures), desc="🚀 Downloading"):
        result = future.result()
        print(result)

# Checked if code is correct 18/09/2024
#(apply clear env() if needed)

In [ ]:
###########################
# Processing Weather Data #
###########################

# === CONFIG ===
data_folder = directory + "Data\\Raw\\Weather\\"
grid_size = 50000  # 50 km in meters
crs_latlon = CRS.from_epsg(4326)
crs_projected = CRS.from_epsg(5070)
vars_to_keep = ['prcp', 'tavg', 'tmax', 'tmin']

transformer = Transformer.from_crs(crs_latlon, crs_projected, always_xy=True)

def process_nc_file(file_path):
    gdfs = []
    ds = xr.open_dataset(file_path)

    vars_in_file = [v for v in vars_to_keep if v in ds.data_vars]
    if not vars_in_file:
        print(f"⚠️ Skipping {file_path}: no expected variables found.")
        return None

    time_len = ds.sizes.get("time", 0)

    # Prepare coordinate grid once
    lat, lon = xr.broadcast(ds['lat'], ds['lon'])   # ✅ correct
    lon_flat = lon.values.ravel()
    lat_flat = lat.values.ravel()
    x_flat, y_flat = transformer.transform(lon_flat, lat_flat)  # ✅ correct


    for t in range(time_len):
        date = pd.to_datetime(ds['time'][t].values).date()

        df = pd.DataFrame({'lat': lat_flat, 'lon': lon_flat, 'x': x_flat, 'y': y_flat})
        df["x_bin"] = (df["x"] // grid_size) * grid_size
        df["y_bin"] = (df["y"] // grid_size) * grid_size
        df["date"] = date

        for var in vars_in_file:
            values = ds[var].isel(time=t).values.ravel()
            df[var] = values

        # Drop rows with NaNs in all variables
        df.dropna(subset=vars_in_file, how="all", inplace=True)

        # Aggregate
        agg_df = df.groupby(["x_bin", "y_bin", "date"])[vars_in_file].mean().reset_index()

        # Create square polygons
        agg_df["geometry"] = [
            box(x, y, x + grid_size, y + grid_size)
            for x, y in zip(agg_df["x_bin"], agg_df["y_bin"])
        ]

        gdf = gpd.GeoDataFrame(agg_df, geometry="geometry", crs=crs_projected)
        gdfs.append(gdf)

    ds.close()
    return pd.concat(gdfs, ignore_index=True) if gdfs else None

# === Master list of all daily GeoDataFrames ===
all_gdfs = []

# === Track errors ===
error_log = []

# === Process all NetCDF files ===
nc_files = [f for f in os.listdir(data_folder) if f.endswith(".nc")]

for file_name in tqdm(nc_files, desc="🚀 Processing NetCDF files"):
    full_path = os.path.join(data_folder, file_name)
    try:
        gdf = process_nc_file(full_path)
        if gdf is not None:
            all_gdfs.append(gdf)
    except Exception as e:
        print(f"❌ Error processing {file_name}: {e}")
        error_log.append({"file": file_name, "error": str(e)})

# === Combine into one final GeoDataFrame ===
if all_gdfs:
    final_gdf_1 = pd.concat(all_gdfs, ignore_index=True)
    print(f"\n🎉 All done! Total rows: {len(final_gdf_1)}")
else:
    print("⚠️ No data was processed.")

# === Save error log ===
if error_log:
    error_df = pd.DataFrame(error_log)
    error_df.to_csv(directory+"Data\\"+"weather_processing_errors.csv", index=False)
    print(f"\n🚨 {len(error_df)} file(s) failed to process. Saved to weather_processing_errors.csv")

# Checked if code is correct 18/09/2024 
#(apply clear env() if needed)

In [ ]:
###########################
# Processing Weather Data #
###########################

import os
import pandas as pd
import geopandas as gpd
import xarray as xr
from pyproj import CRS, Transformer
from shapely.geometry import box
from tqdm import tqdm

# === CONFIG ===
data_folder = directory + "Data\\Raw\\Weather\\"
output_folder = directory + "Data\\Processed\\Weather\\"  # ⭐ NEW: folder for processed files
os.makedirs(output_folder, exist_ok=True)                 # ⭐ NEW: ensure it exists

grid_size = 50000  # 50 km in meters
crs_latlon = CRS.from_epsg(4326)
crs_projected = CRS.from_epsg(5070)
vars_to_keep = ['prcp', 'tavg', 'tmax', 'tmin']

transformer = Transformer.from_crs(crs_latlon, crs_projected, always_xy=True)

def process_nc_file(file_path):
    gdfs = []
    ds = xr.open_dataset(file_path)

    vars_in_file = [v for v in vars_to_keep if v in ds.data_vars]
    if not vars_in_file:
        print(f"⚠️ Skipping {file_path}: no expected variables found.")
        ds.close()
        return None

    time_len = ds.sizes.get("time", 0)

    # Prepare coordinate grid once
    lat, lon = xr.broadcast(ds['lat'], ds['lon'])   # ✅ correct
    lon_flat = lon.values.ravel()
    lat_flat = lat.values.ravel()
    x_flat, y_flat = transformer.transform(lon_flat, lat_flat)  # ✅ correct

    for t in range(time_len):
        date = pd.to_datetime(ds['time'][t].values).date()

        df = pd.DataFrame({'lat': lat_flat, 'lon': lon_flat, 'x': x_flat, 'y': y_flat})
        df["x_bin"] = (df["x"] // grid_size) * grid_size
        df["y_bin"] = (df["y"] // grid_size) * grid_size
        df["date"] = date

        for var in vars_in_file:
            values = ds[var].isel(time=t).values.ravel()
            df[var] = values

        # Drop rows with NaNs in all variables
        df.dropna(subset=vars_in_file, how="all", inplace=True)

        # Aggregate
        agg_df = df.groupby(["x_bin", "y_bin", "date"])[vars_in_file].mean().reset_index()

        # Create square polygons
        agg_df["geometry"] = [
            box(x, y, x + grid_size, y + grid_size)
            for x, y in zip(agg_df["x_bin"], agg_df["y_bin"])
        ]

        gdf = gpd.GeoDataFrame(agg_df, geometry="geometry", crs=crs_projected)
        gdfs.append(gdf)

    ds.close()
    return pd.concat(gdfs, ignore_index=True) if gdfs else None

# === Master list of all daily GeoDataFrames ===
all_gdfs = []

# === Track errors ===
error_log = []

# === Process all NetCDF files ===
nc_files = [f for f in os.listdir(data_folder) if f.endswith(".nc")]

for file_name in tqdm(nc_files, desc="🚀 Processing NetCDF files"):
    full_path = os.path.join(data_folder, file_name)

    # ⭐ NEW: build expected processed filename and skip if it exists
    base_name, _ = os.path.splitext(file_name)
    processed_path = os.path.join(output_folder, base_name + "_processed.parquet")

    if os.path.exists(processed_path):
        print(f"✅ Skipping {file_name}: processed file already exists at {processed_path}")
        continue

    try:
        gdf = process_nc_file(full_path)
        if gdf is not None:
            all_gdfs.append(gdf)

            # ⭐ NEW: save per-file processed result
            gdf.to_parquet(processed_path)
    except Exception as e:
        print(f"❌ Error processing {file_name}: {e}")
        error_log.append({"file": file_name, "error": str(e)})

# === Combine into one final GeoDataFrame (only for files processed in this run) ===
#if all_gdfs:
    #final_gdf_1 = pd.concat(all_gdfs, ignore_index=True)
    #print(f"\n🎉 All done! Total rows (this run only): {len(final_gdf_1)}")
#else:
    #print("⚠️ No new data was processed.")

# === Save error log ===
if error_log:
    error_df = pd.DataFrame(error_log)
    error_df.to_csv(directory + "Data\\" + "weather_processing_errors.csv", index=False)
    print(f"\n🚨 {len(error_df)} file(s) failed to process. Saved to weather_processing_errors.csv")

# Checked if code is correct 18/09/2024 
# (apply clear env() if needed)


In [ ]:
######################################################
# Processing Weather Data - Monthly to 5-year chunks #
######################################################
# This section processes monthly NetCDF files and combines them into 5-year periods

# === CONFIG ===
# Option 2: Forward slashes (Windows handles these fine)
data_folder = directory + "Data\\Raw\\Weather\\"
output_folder = directory + "Data\\Weather\\"
grid_size = 50000  # 50 km in meters
crs_latlon = CRS.from_epsg(4326)
crs_projected = CRS.from_epsg(5070)
vars_to_keep = ['prcp', 'tavg', 'tmax', 'tmin']

# === DEBUG: Check if paths exist and list files ===
print(f"Data folder exists: {os.path.exists(data_folder)}")
print(f"Output folder exists: {os.path.exists(output_folder)}")

if os.path.exists(data_folder):
    all_files = os.listdir(data_folder)
    print(f"Total files in data folder: {len(all_files)}")
    nc_files = [f for f in all_files if f.endswith(".nc")]
    print(f"NetCDF files found: {len(nc_files)}")
    if nc_files:
        print("First 5 NetCDF files:")
        for f in nc_files[:5]:
            print(f"  - {f}")
else:
    print(f"Data folder does not exist: {data_folder}")

# Define 5-year periods
year_periods = [
    (1991, 1995),
    (1996, 2000), 
    (2001, 2005),
    (2006, 2010),
    (2011, 2015),
    (2016, 2020),
    (2021, 2025)  # Adjust the end year as needed
]

transformer = Transformer.from_crs(crs_latlon, crs_projected, always_xy=True)

def process_nc_file(file_path):
    gdfs = []
    ds = xr.open_dataset(file_path)

    vars_in_file = [v for v in vars_to_keep if v in ds.data_vars]
    if not vars_in_file:
        print(f"⚠️ Skipping {file_path}: no expected variables found.")
        return None

    time_len = ds.sizes.get("time", 0)

    # Prepare coordinate grid once
    lat, lon = xr.broadcast(ds['lat'], ds['lon'])
    lon_flat = lon.values.ravel()
    lat_flat = lat.values.ravel()
    x_flat, y_flat = transformer.transform(lon_flat, lat_flat)

    for t in range(time_len):
        date = pd.to_datetime(ds['time'][t].values).date()
        year = date.year

        df = pd.DataFrame({'lat': lat_flat, 'lon': lon_flat, 'x': x_flat, 'y': y_flat})
        df["x_bin"] = (df["x"] // grid_size) * grid_size
        df["y_bin"] = (df["y"] // grid_size) * grid_size
        df["date"] = date
        df["year"] = year

        for var in vars_in_file:
            values = ds[var].isel(time=t).values.ravel()
            df[var] = values

        # Drop rows with NaNs in all variables
        df.dropna(subset=vars_in_file, how="all", inplace=True)

        # Aggregate
        agg_df = df.groupby(["x_bin", "y_bin", "date", "year"])[vars_in_file].mean().reset_index()

        # Create square polygons
        agg_df["geometry"] = [
            box(x, y, x + grid_size, y + grid_size)
            for x, y in zip(agg_df["x_bin"], agg_df["y_bin"])
        ]

        gdf = gpd.GeoDataFrame(agg_df, geometry="geometry", crs=crs_projected)
        gdfs.append(gdf)

    ds.close()
    return pd.concat(gdfs, ignore_index=True) if gdfs else None

# Create output directory if it doesn't exist
os.makedirs(output_folder, exist_ok=True)

# === Process files by 5-year periods ===
nc_files = sorted([f for f in os.listdir(data_folder) if f.endswith(".nc")])

for start_year, end_year in year_periods:
    print(f"\n📊 Processing period {start_year}-{end_year}")
    
    # Filter files for this period
    period_files = []
    for file_name in nc_files:
        # Extract year from filename (assuming format like "nclimgrid_199101.nc")
        try:
            year_part = file_name.split('-')[1][:4]  # Changed from split('_') to split('-')
            file_year = int(year_part)
            if start_year <= file_year <= end_year:
                period_files.append(file_name)
        except (IndexError, ValueError):
            continue
    
    if not period_files:
        print(f"⚠️ No files found for period {start_year}-{end_year}")
        continue
    
    # Process all files for this period
    period_gdfs = []
    error_log_period = []
    
    for file_name in tqdm(period_files, desc=f"🚀 Processing {start_year}-{end_year}"):
        full_path = os.path.join(data_folder, file_name)
        try:
            gdf = process_nc_file(full_path)
            if gdf is not None:
                period_gdfs.append(gdf)
        except Exception as e:
            print(f"❌ Error processing {file_name}: {e}")
            error_log_period.append({"file": file_name, "error": str(e)})
    
    # Combine and save the 5-year period data
    if period_gdfs:
        final_gdf_period = pd.concat(period_gdfs, ignore_index=True)
        output_filename = f"weather_data_{start_year}_{end_year}.gpkg"
        output_path = os.path.join(output_folder, output_filename)
        
        # Save as GeoPackage (more efficient than shapefile for large datasets)
        final_gdf_period.to_file(output_path, driver='GPKG')
        print(f"💾 Saved {len(final_gdf_period)} rows to {output_filename}")
        
        # Also save as CSV for easy analysis (without geometry)
        csv_output_path = os.path.join(output_folder, f"weather_data_{start_year}_{end_year}.csv")
        final_gdf_period.to_csv(csv_output_path, index=False)
        print(f"💾 Saved CSV version to {csv_output_path.split('/')[-1]}")
    else:
        print(f"⚠️ No data processed for period {start_year}-{end_year}")
    
    # Save error log for this period
    if error_log_period:
        error_df = pd.DataFrame(error_log_period)
        error_output_path = os.path.join(output_folder, f"error_log_{start_year}_{end_year}.csv")
        error_df.to_csv(error_output_path, index=False)
        print(f"🚨 {len(error_df)} file(s) failed to process for this period")

print("\n🎉 All processing complete!")

# Checked if code is correct 18/09/2024 
#(apply clear env() if needed)


In [ ]:
#########################################
# Creation of Daily Temperature Dataset # (Start of Creation of Temperature Dataset, previous sections cleans and downloads data)
#########################################
# (i) Importing Required Datasets
# Load files
nclimDaily_a = pd.read_csv(directory + "Data\\Weather\\weather_data_2021_2025.csv")
nclimDaily_b = pd.read_csv(directory + "Data\\Weather\\\\weather_data_2016_2020.csv")

# Append (concatenate)
nclimDaily = pd.concat([nclimDaily_a, nclimDaily_b], ignore_index=True)

# Make sure your date column is parsed as datetime
nclimDaily['date'] = pd.to_datetime(nclimDaily['date'])

# Filter 2020–2025
nclimDaily = nclimDaily[(nclimDaily['date'].dt.year >= 2020) & (nclimDaily['date'].dt.year <= 2025)]
# Create geometry column from WKT
nclimDaily['geometry'] = nclimDaily['geometry'].apply(wkt.loads)
nclimDaily = gpd.GeoDataFrame(nclimDaily, geometry='geometry')


US_county_shp = gpd.read_file(directory+'Data\\Raw\\US Counties\\US_County_SHP\\c_05mr24.shp')
US_county_shp = (
    US_county_shp.groupby('FIPS')
    .agg({
        'STATE': 'first',
        'CWA': 'first',
        'COUNTYNAME': 'first',
        'TIME_ZONE': 'first',
        'FE_AREA': 'first',
        'LON': 'mean',
        'LAT': 'mean',
        'geometry': lambda x: unary_union(list(x))
    })
    .reset_index()
)
US_county_shp = gpd.GeoDataFrame(US_county_shp, geometry='geometry', crs="EPSG:4326")
US_county_shp = US_county_shp.to_crs("EPSG:5070")
nclimDaily = nclimDaily.set_crs("EPSG:5070")

# 1. Defining Contiguous U.S. Bounds
# These bounds are for the contiguous United States, excluding Alaska and Hawaii.
contiguous_bounds = [-2453691.2025, 171085.9341, 2365756.5659, 3381509.2103]
contiguous = US_county_shp.cx[contiguous_bounds[0]:contiguous_bounds[2],contiguous_bounds[1]:contiguous_bounds[3]]
contiguous_nclimDaily = nclimDaily.cx[contiguous_bounds[0]:contiguous_bounds[2],contiguous_bounds[1]:contiguous_bounds[3]]

# --- 0) Master weather list (unique points in a fixed order)
# Use your stable key, e.g. 'weather_id' or 'STNID' (must exist in the daily table)
# --- neighbours & power
k = 4        # number of neighbours
p = 2        # IDW power

# --- 1) Create a stable weather_id if you don't already have one
if 'weather_id' not in contiguous_nclimDaily.columns:
    contiguous_nclimDaily = contiguous_nclimDaily.copy()
    contiguous_nclimDaily['weather_id'] = contiguous_nclimDaily.geometry.apply(lambda g: g.wkb_hex)
key = 'weather_id'  # or 'STNID'
weather_master = (
    contiguous_nclimDaily[[key, 'geometry']]
    .drop_duplicates(subset=key)
    .reset_index(drop=True)
)

# Build coordinates from this master (EPSG:5070 -> use (x, y))
weather_xy = np.vstack(weather_master.geometry.centroid.apply(lambda g: (g.x, g.y)).to_numpy())

# --- 1) Counties and BallTree (already in EPSG:5070)
county_xy = np.vstack(contiguous.geometry.centroid.apply(lambda g: (g.x, g.y)).to_numpy())
tree = BallTree(weather_xy, metric='euclidean')
dist_m, idx = tree.query(county_xy, k=k)  # <-- idx indexes into weather_master


# 5. IDW weights (handle zero-distance safely)
zero_mask = (dist_m == 0)
if zero_mask.any():
    weights = np.zeros_like(dist_m, dtype=float)
    row_has_zero = zero_mask.any(axis=1)
    weights[row_has_zero] = zero_mask[row_has_zero].astype(float)  # 1 for exact match, else 0
else:
    weights = 1.0 / np.power(dist_m, p)

# normalise rows
weights /= weights.sum(axis=1, keepdims=True)

# --- 2) Daily IDW using the same master order
order = weather_master[key].to_numpy()

nearest_idx  = idx[:, 0]                               # nearest neighbour index (into weather_master)
nearest_id   = weather_master[key].to_numpy()[nearest_idx]
nearest_dist = dist_m[:, 0]                            # metres

out = []
for date, g in contiguous_nclimDaily.groupby('date', sort=True):
    # Align day to master weather order
    gg = g.set_index(key).reindex(order)

    # (n_weather, 4)
    vals = gg[['tavg', 'tmax', 'tmin', 'prcp']].to_numpy()

    # Gather each county’s k neighbours: (n_counties, k, 4)
    vals_neigh = vals[idx]

    # ---------- IDW (ignore missing neighbours, renormalise) ----------
    valid = ~np.isnan(vals_neigh)                         # (n_counties, k, 4)
    w = weights[..., None] * valid                        # (n_counties, k, 4)
    w_sum = w.sum(axis=1, keepdims=True)                  # (n_counties, 1, 4)
    w_norm = np.divide(w, w_sum, where=(w_sum > 0))       # avoid /0
    idw = (vals_neigh * w_norm).sum(axis=1)               # (n_counties, 4)

    # ---------- Nearest neighbour (with fallback if nearest is NaN) ----------
    # Start with nearest; if NaN, take next nearest (2nd, 3rd, ...)
    nn = np.full((idx.shape[0], 4), np.nan)               # (n_counties, 4)
    for j in range(idx.shape[1]):                         # j over neighbours in ascending distance
        # fill where still NaN and neighbour j has a value
        mask = np.isnan(nn) & ~np.isnan(vals_neigh[:, j, :])
        nn[mask] = vals_neigh[:, j, :][mask]

    out.append(pd.DataFrame({
    'FIPS': contiguous['FIPS'].values,
    'date': date,
    'nearest_weather_id': nearest_id,
    'nearest_dist_m': np.round(nearest_dist, 2),  # round distance too if you want

    # IDW
    'idw_tavg': np.round(idw[:, 0], 2),
    'idw_tmax': np.round(idw[:, 1], 2),
    'idw_tmin': np.round(idw[:, 2], 2),
    'idw_prcp': np.round(idw[:, 3], 2),

    # Nearest neighbour (fallback)
    'nn_tavg': np.round(nn[:, 0], 2),
    'nn_tmax': np.round(nn[:, 1], 2),
    'nn_tmin': np.round(nn[:, 2], 2),
    'nn_prcp': np.round(nn[:, 3], 2),
    }))
weighted_weather_daily = pd.concat(out, ignore_index=True)

# === Save the final DataFrame ===
weighted_weather_daily.to_csv(directory+"Data\\nclim_county_matched.csv", index=False)
# Checked if code is correct 18/09/2024 

clear_env(keep=["directory"])

In [ ]:
####################################
#Creation of Climate Normal Dataset#
####################################
weather_data_path = directory + "Data\\Weather\\"

# Get only CSV files starting with 'weather_data_'
csv_files = [f for f in os.listdir(weather_data_path) if f.endswith('.csv') and f.startswith('weather_data_')]

# Sort the files to process in order
csv_files.sort()

print(f"Found {len(csv_files)} files to process:")
for file in csv_files:
    print(f"  - {file}")

# Append (concatenate) all CSV files
data_frames = []
for csv_file in csv_files:
    file_path = os.path.join(weather_data_path, csv_file)
    df = pd.read_csv(file_path)
    data_frames.append(df)
nclimDaily_1991_2020 = pd.concat(data_frames, ignore_index=True)

# Make sure your date column is parsed as datetime
# assuming your nclimDaily_1991_2020 is already a GeoDataFrame
# with columns: ['grid_id', 'date', 'temp', 'geometry']

nclimDaily_1991_2020['date'] = pd.to_datetime(nclimDaily_1991_2020['date'])
nclimDaily_1991_2020['year'] = nclimDaily_1991_2020['date'].dt.year
nclimDaily_1991_2020['month'] = nclimDaily_1991_2020['date'].dt.month

# filter to 1991–2020
nclimDaily_1991_2020 = nclimDaily_1991_2020[
    (nclimDaily_1991_2020['year'] >= 1991) &
    (nclimDaily_1991_2020['year'] <= 2020)
]

# use centroid coordinates to define grid_id
# create a grid_id by combining lat/lon into a string
nclimDaily_1991_2020['grid_id'] = (
    nclimDaily_1991_2020['x_bin'].astype(str) + "_" + nclimDaily_1991_2020['y_bin'].astype(str)
)

# aggregate: daily mean + SD for each grid cell
nclimDaily_1991_2020['doy'] = nclimDaily_1991_2020['date'].dt.dayofyear
Normal_doy_1991_2020 = (
    nclimDaily_1991_2020
    .groupby(['grid_id', 'doy'])
    .agg(mean_tavg=('tavg', 'mean'),
         sd_tavg=('tavg', 'std'),
         mean_max=('tmax', 'mean'),
         sd_max=('tmax', 'std'),
         mean_min=('tmin', 'mean'),
         sd_min=('tmin', 'std'),
         )
    .reset_index()
)

# bring back the geometry (merge on grid_id)
Normal_doy_1991_2020 = Normal_doy_1991_2020.merge(
    nclimDaily_1991_2020[['grid_id', 'geometry']].drop_duplicates(),
    on='grid_id',
    how='left'
)

Normal_doy_1991_2020.to_csv(directory+"Data\\nclim_Normal_Daily_1991_2020.csv", index=False)


# aggregate: monthly mean + SD for each grid cell
Normal_Month_1991_2020 = (
    nclimDaily_1991_2020
    .groupby(['grid_id', 'month'])
    .agg(mean_tavg=('tavg', 'mean'),
         sd_tavg=('tavg', 'std'),
         mean_max=('tmax', 'mean'),
         sd_max=('tmax', 'std'),
         mean_min=('tmin', 'mean'),
         sd_min=('tmin', 'std'),
         )
    .reset_index()
)

# bring back the geometry (merge on grid_id)
Normal_Month_1991_2020 = Normal_Month_1991_2020.merge(
    nclimDaily_1991_2020[['grid_id', 'geometry']].drop_duplicates(),
    on='grid_id',
    how='left'
)

# === Save the final DataFrame ===
Normal_Month_1991_2020.to_csv(directory+"Data\\nclim_Normal_Month_1991_2020.csv", index=False)
# Checked if code is correct 24/09/2024 
clear_env(keep=["directory", "Normal_Month_1991_2020"])

In [ ]:
###############################################
#Creation of Climate Normal County-Month Panel#
###############################################
nclimNormal = pd.read_csv(directory+"Data\\nclim_Normal_Month_1991_2020.csv")
nclimNormal['geometry'] = nclimNormal['geometry'].apply(wkt.loads)
nclimNormal = gpd.GeoDataFrame(nclimNormal, geometry='geometry')

US_county_shp = gpd.read_file(directory+'Data\\Raw\\US Counties\\US_County_SHP\\c_05mr24.shp')
US_county_shp = (
    US_county_shp.groupby('FIPS')
    .agg({
        'STATE': 'first',
        'CWA': 'first',
        'COUNTYNAME': 'first',
        'TIME_ZONE': 'first',
        'FE_AREA': 'first',
        'LON': 'mean',
        'LAT': 'mean',
        'geometry': lambda x: unary_union(list(x))
    })
    .reset_index()
)
US_county_shp = gpd.GeoDataFrame(US_county_shp, geometry='geometry', crs="EPSG:4326")
US_county_shp = US_county_shp.to_crs("EPSG:5070")

US_county_shp = US_county_shp.to_crs("EPSG:5070")
nclimNormal = nclimNormal.set_crs("EPSG:5070")

# 1. Defining Contiguous U.S. Bounds
# These bounds are for the contiguous United States, excluding Alaska and Hawaii.
contiguous_bounds = [-2453691.2025, 171085.9341, 2365756.5659, 3381509.2103]
contiguous = US_county_shp.cx[contiguous_bounds[0]:contiguous_bounds[2],contiguous_bounds[1]:contiguous_bounds[3]]
contiguous_nclimNormal = nclimNormal.cx[contiguous_bounds[0]:contiguous_bounds[2],contiguous_bounds[1]:contiguous_bounds[3]]

# 1. Defining Contiguous U.S. Bounds
# These bounds are for the contiguous United States, excluding Alaska and Hawaii.
contiguous_bounds = [-2453691.2025, 171085.9341, 2365756.5659, 3381509.2103]
contiguous = US_county_shp.cx[contiguous_bounds[0]:contiguous_bounds[2],contiguous_bounds[1]:contiguous_bounds[3]]
contiguous_nclimNormal = nclimNormal.cx[contiguous_bounds[0]:contiguous_bounds[2],contiguous_bounds[1]:contiguous_bounds[3]]

# --- 0) Master weather list (unique points in a fixed order)
# Use your stable key, e.g. 'weather_id' or 'STNID' (must exist in the daily table)
# --- neighbours & power
k = 4        # number of neighbours
p = 2        # IDW power

# --- 1) Create a stable weather_id if you don't already have one
if 'weather_id' not in contiguous_nclimNormal.columns:
    contiguous_nclimNormal = contiguous_nclimNormal.copy()
    contiguous_nclimNormal['weather_id'] = contiguous_nclimNormal.geometry.apply(lambda g: g.wkb_hex)
key = 'weather_id'  # or 'STNID'
weather_master = (
    contiguous_nclimNormal[[key, 'geometry']]
    .drop_duplicates(subset=key)
    .reset_index(drop=True)
)

# Build coordinates from this master (EPSG:5070 -> use (x, y))
weather_xy = np.vstack(weather_master.geometry.centroid.apply(lambda g: (g.x, g.y)).to_numpy())

# --- 1) Counties and BallTree (already in EPSG:5070)
county_xy = np.vstack(contiguous.geometry.centroid.apply(lambda g: (g.x, g.y)).to_numpy())
tree = BallTree(weather_xy, metric='euclidean')
dist_m, idx = tree.query(county_xy, k=k)  # <-- idx indexes into weather_master


# 5. IDW weights (handle zero-distance safely)
zero_mask = (dist_m == 0)
if zero_mask.any():
    weights = np.zeros_like(dist_m, dtype=float)
    row_has_zero = zero_mask.any(axis=1)
    weights[row_has_zero] = zero_mask[row_has_zero].astype(float)  # 1 for exact match, else 0
else:
    weights = 1.0 / np.power(dist_m, p)

# normalise rows
weights /= weights.sum(axis=1, keepdims=True)

# --- 2) Daily IDW using the same master order
order = weather_master[key].to_numpy()

nearest_idx  = idx[:, 0]                               # nearest neighbour index (into weather_master)
nearest_id   = weather_master[key].to_numpy()[nearest_idx]
nearest_dist = dist_m[:, 0]                            # metres

out = []
for date, g in contiguous_nclimNormal.groupby('month', sort=True):
    # Align day to master weather order
    gg = g.set_index(key).reindex(order)

    # Six stats: (mean_tavg, sd_tavg, mean_max, sd_max, mean_min, sd_min)
    vals = gg[['mean_tavg', 'sd_tavg', 'mean_max', 'sd_max', 'mean_min', 'sd_min']].to_numpy()

    # Gather each county’s k neighbours -> (n_counties, k, 6)
    vals_neigh = vals[idx]

    # ---------- IDW (ignore missing neighbours, renormalise) ----------
    # weights: (n_counties, k) -> broadcast to (..., 1)
    valid = ~np.isnan(vals_neigh)                          # (n_counties, k, 6)
    w = weights[..., None] * valid                         # (n_counties, k, 6)
    num = (vals_neigh * w).sum(axis=1)                     # (n_counties, 6)
    den = w.sum(axis=1)                                    # (n_counties, 6)
    idw = np.divide(num, den, out=np.full_like(num, np.nan), where=(den > 0))  # (n_counties, 6)

    # ---------- Nearest neighbour (with fallback if nearest is NaN) ----------
    n_counties = contiguous.shape[0]
    n_stats = vals_neigh.shape[2]                          # 6
    nn = np.full((n_counties, n_stats), np.nan)            # (n_counties, 6)

    # Fill from nearest to farther neighbours, per element
    for j in range(idx.shape[1]):
        curr = vals_neigh[:, j, :]                         # (n_counties, 6)
        mask = np.isnan(nn) & ~np.isnan(curr)
        nn[mask] = curr[mask]

    out.append(pd.DataFrame({
        'FIPS': contiguous['FIPS'].values,
        'month': date,
        'nearest_weather_id': nearest_id,       # keep your existing computation
        'nearest_dist_m': np.round(nearest_dist, 2),

        # ---- IDW mapped correctly ----
        'idw_mean_tavg': np.round(idw[:, 0], 2),
        'idw_sd_tavg'  : np.round(idw[:, 1], 2),
        'idw_mean_max' : np.round(idw[:, 2], 2),
        'idw_sd_max'   : np.round(idw[:, 3], 2),
        'idw_mean_min' : np.round(idw[:, 4], 2),
        'idw_sd_min'   : np.round(idw[:, 5], 2),

        # ---- Nearest neighbour mapped correctly ----
        'nn_mean_tavg' : np.round(nn[:, 0], 2),
        'nn_sd_tavg'   : np.round(nn[:, 1], 2),
        'nn_mean_max'  : np.round(nn[:, 2], 2),
        'nn_sd_max'    : np.round(nn[:, 3], 2),
        'nn_mean_min'  : np.round(nn[:, 4], 2),
        'nn_sd_min'    : np.round(nn[:, 5], 2),
    }))

weighted_weather_normal = pd.concat(out, ignore_index=True)

# === Save the final GeoDataFrame ===
weighted_weather_normal.to_csv(directory+"Data\\county_month_normal_1991_2020.csv", index=False)

# Checked if code is correct 24/09/2024 
clear_env(keep=["directory"])


In [ ]:
#############################################
#Creation of Climate Normal County-Day Panel#
#############################################
nclimNormal = pd.read_csv(directory+"Data\\nclim_Normal_Daily_1991_2020.csv")
nclimNormal['geometry'] = nclimNormal['geometry'].apply(wkt.loads)
nclimNormal = gpd.GeoDataFrame(nclimNormal, geometry='geometry')

US_county_shp = gpd.read_file(directory+'Data\\Raw\\US Counties\\US_County_SHP\\c_05mr24.shp')
US_county_shp = (
    US_county_shp.groupby('FIPS')
    .agg({
        'STATE': 'first',
        'CWA': 'first',
        'COUNTYNAME': 'first',
        'TIME_ZONE': 'first',
        'FE_AREA': 'first',
        'LON': 'mean',
        'LAT': 'mean',
        'geometry': lambda x: unary_union(list(x))
    })
    .reset_index()
)
US_county_shp = gpd.GeoDataFrame(US_county_shp, geometry='geometry', crs="EPSG:4326")
US_county_shp = US_county_shp.to_crs("EPSG:5070")

US_county_shp = US_county_shp.to_crs("EPSG:5070")
nclimNormal = nclimNormal.set_crs("EPSG:5070")

# 1. Defining Contiguous U.S. Bounds
# These bounds are for the contiguous United States, excluding Alaska and Hawaii.
contiguous_bounds = [-2453691.2025, 171085.9341, 2365756.5659, 3381509.2103]
contiguous = US_county_shp.cx[contiguous_bounds[0]:contiguous_bounds[2],contiguous_bounds[1]:contiguous_bounds[3]]
contiguous_nclimNormal = nclimNormal.cx[contiguous_bounds[0]:contiguous_bounds[2],contiguous_bounds[1]:contiguous_bounds[3]]

# 1. Defining Contiguous U.S. Bounds
# These bounds are for the contiguous United States, excluding Alaska and Hawaii.
contiguous_bounds = [-2453691.2025, 171085.9341, 2365756.5659, 3381509.2103]
contiguous = US_county_shp.cx[contiguous_bounds[0]:contiguous_bounds[2],contiguous_bounds[1]:contiguous_bounds[3]]
contiguous_nclimNormal = nclimNormal.cx[contiguous_bounds[0]:contiguous_bounds[2],contiguous_bounds[1]:contiguous_bounds[3]]

# --- 0) Master weather list (unique points in a fixed order)
# Use your stable key, e.g. 'weather_id' or 'STNID' (must exist in the daily table)
# --- neighbours & power
k = 4        # number of neighbours
p = 2        # IDW power

# --- 1) Create a stable weather_id if you don't already have one
if 'weather_id' not in contiguous_nclimNormal.columns:
    contiguous_nclimNormal = contiguous_nclimNormal.copy()
    contiguous_nclimNormal['weather_id'] = contiguous_nclimNormal.geometry.apply(lambda g: g.wkb_hex)
key = 'weather_id'  # or 'STNID'
weather_master = (
    contiguous_nclimNormal[[key, 'geometry']]
    .drop_duplicates(subset=key)
    .reset_index(drop=True)
)

# Build coordinates from this master (EPSG:5070 -> use (x, y))
weather_xy = np.vstack(weather_master.geometry.centroid.apply(lambda g: (g.x, g.y)).to_numpy())

# --- 1) Counties and BallTree (already in EPSG:5070)
county_xy = np.vstack(contiguous.geometry.centroid.apply(lambda g: (g.x, g.y)).to_numpy())
tree = BallTree(weather_xy, metric='euclidean')
dist_m, idx = tree.query(county_xy, k=k)  # <-- idx indexes into weather_master


# 5. IDW weights (handle zero-distance safely)
zero_mask = (dist_m == 0)
if zero_mask.any():
    weights = np.zeros_like(dist_m, dtype=float)
    row_has_zero = zero_mask.any(axis=1)
    weights[row_has_zero] = zero_mask[row_has_zero].astype(float)  # 1 for exact match, else 0
else:
    weights = 1.0 / np.power(dist_m, p)

# normalise rows
weights /= weights.sum(axis=1, keepdims=True)

# --- 2) Daily IDW using the same master order
order = weather_master[key].to_numpy()

nearest_idx  = idx[:, 0]                               # nearest neighbour index (into weather_master)
nearest_id   = weather_master[key].to_numpy()[nearest_idx]
nearest_dist = dist_m[:, 0]                            # metres

out = []
for date, g in contiguous_nclimNormal.groupby('doy', sort=True):
    # Align day to master weather order
    gg = g.set_index(key).reindex(order)

    # Two variables: mean and sd  -> shape (n_weather, 2)
    vals = gg[['mean_tavg', 'sd_tavg']].to_numpy()

    # Gather each county’s k neighbours -> (n_counties, k, 2)
    vals_neigh = vals[idx]

    # ---------- IDW (ignore missing neighbours, renormalise) ----------
    # weights shape: (n_counties, k)
    # broadcast to (n_counties, k, 2)
    valid = ~np.isnan(vals_neigh)                       # (n_counties, k, 2)
    w = weights[..., None] * valid                      # (n_counties, k, 2), zero-out missing
    num = (vals_neigh * w).sum(axis=1)                  # (n_counties, 2)
    den = w.sum(axis=1)                                 # (n_counties, 2)
    idw = np.divide(num, den, out=np.full_like(num, np.nan), where=(den > 0))

    # ---------- Nearest neighbour (with fallback if nearest is NaN) ----------
    # Start with all-NaN (n_counties, 2) to hold [mean, sd]
    nn = np.full((idx.shape[0], 2), np.nan)

    # loop neighbours in ascending distance; fill first available value per variable
    for j in range(idx.shape[1]):
        curr = vals_neigh[:, j, :]                      # (n_counties, 2)
        nn = np.where(np.isnan(nn) & ~np.isnan(curr), curr, nn)

    out.append(pd.DataFrame({
        'FIPS': contiguous['FIPS'].values,
        'doy': date,
        'nearest_weather_id': nearest_id,       # assuming you computed this already
        'nearest_dist_m': np.round(nearest_dist, 2),

        # IDW
        'idw_mean_tavg': np.round(idw[:, 0], 2),
        'idw_sd_tavg'  : np.round(idw[:, 1], 2),

        # Nearest neighbour (fallback)
        'nn_mean_tavg': np.round(nn[:, 0], 2),
        'nn_sd_tavg'  : np.round(nn[:, 1], 2),
    }))

weighted_weather_normal = pd.concat(out, ignore_index=True)

# === Save the final GeoDataFrame ===
weighted_weather_normal.to_csv(directory+"Data\\county_daily_normal_1991_2020.csv", index=False)

# Checked if code is correct 24/09/2024 
clear_env(keep=["directory"])

In [2]:
#####################################
# Identification of Climate Regions #
#####################################
weighted_weather_normal = pd.read_csv(directory + "Data\\county_daily_normal_1991_2020.csv")
#Calculating cooling degree days for each unique grid
#weighted_weather_normal['dlytavg_f'] = (weighted_weather_normal['nn_dlytavg_norm'] * 1.8) + 32
weighted_weather_normal['CDD'] = (weighted_weather_normal['nn_mean_tavg'] - 18).clip(lower=0)

cdd_summed = weighted_weather_normal.groupby("FIPS", as_index=False)["CDD"].sum()
cdd_summed = cdd_summed.rename(columns={"CDD": "total_CDD"})

# We want groups labelled 1, 2, but allow qcut to drop duplicates safely
cdd_summed["CDD_median"], bins = pd.qcut(
    cdd_summed["total_CDD"],
    q=2,
    labels=False,      # get integer codes: 0, 1, ...
    retbins=True,
    duplicates="drop"
)

# Shift codes so groups are 1, 2, ... (typically 1 and 2 for a median split)
cdd_summed["CDD_median"] = cdd_summed["CDD_median"].astype("float")
cdd_summed["CDD_median"] = cdd_summed["CDD_median"] + 1

print("Median boundaries (bin edges):", bins)
print(cdd_summed["CDD_median"].value_counts())

quartile_labels = [1, 2, 3, 4]
cdd_summed["CDD_quartile"], bins = pd.qcut(
    cdd_summed["total_CDD"], 
    q=4, 
    labels=quartile_labels, 
    retbins=True,
    duplicates="drop"  # Just in case there are tied values
)

# Print the cut points
print("Median boundaries (bin edges):", bins)

# === Save the final GeoDataFrame ===
cdd_summed.to_csv(directory+"Data\\county_climate_region.csv", index=False)

# Checked if code is correct 24/09/2024 
clear_env(keep=["directory", "cdd_summed"])

Median boundaries (bin edges): [   0.    582.92 2481.68]
CDD_median
1.0    1561
2.0    1547
Name: count, dtype: int64
Median boundaries (bin edges): [   0.     290.53   582.92   959.475 2481.68 ]


In [ ]:
#############################
# Maping of Climate Regions #
#############################

import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import matplotlib.patches as mpatches

# Matching Map and Climate Tercile Data
Climate_region = gpd.read_file(directory+'Data\\Raw\\US Counties\\US_County_SHP\\c_05mr24.shp')  # Replace with your path
cdd_summed = pd.read_csv(directory + "Data\\county_climate_region.csv")
Climate_region = Climate_region.to_crs("EPSG:5070")
Climate_region['FIPS'] = pd.to_numeric(Climate_region['FIPS'], errors='coerce')
Climate_region = Climate_region.merge(cdd_summed, on='FIPS')
# Define geographic bounding boxes for different regions (EPSG:5070) #Use GIS to identify the correct bounding boxes!
contiguous_bounds = [-2453691.2025, 171085.9341, 2365756.5659, 3381509.2103]

# Separate the regions based on bounding boxes
contiguous = Climate_region.cx[contiguous_bounds[0]:contiguous_bounds[2],contiguous_bounds[1]:contiguous_bounds[3]]

# Custom colours: 1=blue, 2=orange, 3=red
cmap = ListedColormap(["#1f77b4", "#d62728"])

# Ensure CDD_tertile is categorical
contiguous['CDD_median'] = contiguous['CDD_median'].astype(int)

# Create figure and axis
fig, ax = plt.subplots(figsize=(12, 8))

# Plot counties with custom colours
contiguous.plot(
    ax=ax,
    column='CDD_median',
    cmap=cmap,
    edgecolor='black',
    linewidth=0.3,
    alpha=0.8,
    legend=False   # We'll make a custom legend
)

# Create custom legend patches
legend_labels = {1: "Below Median CDD", 2: "Above Median CDD"}
handles = [mpatches.Patch(color=cmap.colors[i-1], label=legend_labels[i]) for i in sorted(legend_labels)]

# Add custom legend
ax.legend(handles=handles, title="CDD Terciles", loc='lower left', fontsize=10, title_fontsize=11, frameon=True)

# Title and formatting
#ax.set_title("Counties Coloured by CDD Terciles", fontsize=16)
ax.axis('off')

plt.savefig(directory + "Results_2/climate_regions.png")
plt.show()

# Merge county geometry to totals so we can inspect geography
contiguous['FIPS'] = contiguous['FIPS'].astype('Int64')
cdd_summed['FIPS'] = cdd_summed['FIPS'].astype('Int64')
tot = contiguous[['FIPS','geometry']].merge(cdd_summed, on='FIPS', how='inner')
tot['x'] = tot.geometry.centroid.x
tot['y'] = tot.geometry.centroid.y

print("Tercile counts:\n", tot['CDD_median'].value_counts(dropna=False))
print("Corr(total_CDD, x):", tot['total_CDD'].corr(tot['x']))
print("Corr(total_CDD, y):", tot['total_CDD'].corr(tot['y']))

# Ensure it's int for plotting
contiguous["CDD_quartile"] = contiguous["CDD_quartile"].astype("Int64")

# -------------------------------------------------------------------
# 3. Build a discrete 4-colour colormap
# -------------------------------------------------------------------
quartile_colors = [
    "#4575b4",  # 1st quartile (coolest)
    "#91bfdb",  # 2nd quartile
    "#fc8d59",  # 3rd quartile
    "#d73027"   # 4th quartile (hottest)
]

quartile_cmap = ListedColormap(quartile_colors)

# -------------------------------------------------------------------
# 4. Plot the map
# -------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(12, 8))

contiguous.plot(
    ax=ax,
    column="CDD_quartile",
    cmap=quartile_cmap,
    edgecolor="black",
    linewidth=0.3,
    alpha=0.8,
    legend=False  # we make a custom legend
)

# -------------------------------------------------------------------
# 5. Custom legend (uses only the quartiles actually present)
# -------------------------------------------------------------------
legend_labels = {
    1: "1st quartile (coolest)",
    2: "2nd quartile",
    3: "3rd quartile",
    4: "4th quartile (hottest)"
}

present_quartiles = sorted(
    int(q) for q in contiguous["CDD_quartile"].dropna().unique()
)

handles = [
    mpatches.Patch(
        color=quartile_colors[q - 1],
        label=legend_labels[q]
    )
    for q in present_quartiles
]

ax.legend(
    handles=handles,
    title="CDD Quartiles",
    loc="lower left",
    fontsize=10,
    title_fontsize=11,
    frameon=True
)

ax.axis("off")

# -------------------------------------------------------------------
# 6. Save and show
# -------------------------------------------------------------------
out_path = os.path.join(directory, r"Results_2", "climate_regions_quartiles.png")
os.makedirs(os.path.dirname(out_path), exist_ok=True)

plt.savefig(out_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved figure to:", out_path)

# Checked if code is correct 24/09/2024 
clear_env(keep=["directory"])

In [3]:
#########################################################################
# Matching of Daily Weather, Climate Normals & Climate Regions Datasets #
#########################################################################
# (A) Daily Weather
Weather_Daily = pd.read_csv(directory+"Data\\nclim_county_matched.csv")
selected_columns_daily = ['FIPS', 'date', 'nn_tavg', 'nn_prcp', 'idw_tavg', 'idw_prcp']
Weather_Daily = Weather_Daily[selected_columns_daily]
Weather_Daily['date'] = pd.to_datetime(Weather_Daily['date'])
Weather_Daily['month'] = Weather_Daily['date'].dt.month
Weather_Daily['doy'] = Weather_Daily['date'].dt.dayofyear

# (B) Climate Normals (Month)
#Weather_Normal = pd.read_csv(directory+"Data\\county_month_normal_1991_2020.csv")
#selected_columns_normal = ['FIPS', 'nn_mean_tavg', 'nn_sd_tavg', 'nn_mean_max', 'nn_sd_max', 'nn_mean_min', 'nn_sd_min', 'month']
#Weather_Normal = Weather_Normal[selected_columns_normal]

# (C) Climate Regions
Climate_region = pd.read_csv(directory + "Data\\county_climate_region.csv")
selected_columns_region = ['FIPS', 'total_CDD']
Climate_region = Climate_region[selected_columns_region]

# Merging of (A), (B) and (C) Datasets
# Merging the DataFrames on 'FIPS'
#Weather = Weather_Daily.merge(Weather_Normal, on=['FIPS', 'month'], suffixes=('_daily', '_normal')) #Fix the ISSUE WITH A LEAP YEAR
Weather = Weather_Daily.merge(Climate_region, on='FIPS')
# Checked if code is correct 24/09/2024 
clear_env(keep=["directory", "Weather"])

In [6]:
######################################################
# Creation of Absolute and Relative Temperature Bins #
######################################################
variables = ["nn_tavg", "idw_tavg"]

for var in variables:
    # 1. Convert to Fahrenheit
    fahr_col = f"{var}_fahrenheit"
    Weather[fahr_col] = (Weather[var] * 1.8) + 32
    
    # 2. Get min and max
    min_value = Weather[fahr_col].min()
    max_value = Weather[fahr_col].max()
    
    # 3. Create 0.5°F bins
    for num in np.arange(np.floor(min_value * 2) / 2,
                         np.ceil(max_value * 2) / 2 + 0.1, 0.5):
        col_name = f"{var}_{num:.1f}"
        Weather[col_name] = (np.floor(Weather[fahr_col] * 2) / 2 == num).astype(int)
    
    # 4. Create 5°F summary bins
    Five_F_degree_bins = np.arange(10, 95, 5)
    for start in Five_F_degree_bins[:-1]:
        end = start + 5
        cols_to_sum = [
            f"{var}_{sd:.1f}"
            for sd in np.arange(start, end, 0.5)
            if f"{var}_{sd:.1f}" in Weather.columns
        ]
        if cols_to_sum:
            bin_col = f"{var}_{int(start)}_{int(end)}".replace("-", "m")
            Weather[bin_col] = Weather[cols_to_sum].sum(axis=1)
    
    # 5. Bin for temps below 10°F
    below_cols = [
        f"{var}_{temp:.1f}"
        for temp in np.arange(-35, 10, 0.5)
        if f"{var}_{temp:.1f}" in Weather.columns
    ]
    Weather[f"{var}_below_10"] = Weather[below_cols].sum(axis=1)
    
    # 6. Bin for temps above 95°F
    above_cols = [
        f"{var}_{temp:.1f}"
        for temp in np.arange(90, 100, 0.5)
        if f"{var}_{temp:.1f}" in Weather.columns
    ]
    Weather[f"{var}_above_90"] = Weather[above_cols].sum(axis=1)


# Selecting Important Columns
selected_columns = ["date", "nn_prcp", 'nn_tavg', "FIPS"]

for var in variables:
    # Add below-10°F bin
    selected_columns.append(f"{var}_below_10")
    
    # Add 5°F bins from 10°F to 95°F
    for start in range(10, 90, 5):
        end = start + 5
        selected_columns.append(f"{var}_{int(start)}_{int(end)}")
    
    # Add above-90°F bin
    selected_columns.append(f"{var}_above_90")

# Add other fixed columns at the end
selected_columns.extend(["total_CDD", "idw_tavg", "idw_prcp"])

# Select from Weather DataFrame
Weather = Weather[selected_columns]

# Variables to apply binning to
vars_to_bin = ["nn_prcp", "idw_prcp"]

# Define bins
rain_bins = [0, 1, 10, 20, float("inf")]

for var in vars_to_bin:
    # Create labels dynamically for each variable
    rain_labels = [
        f"{var}_0_1",
        f"{var}_1_10",
        f"{var}_10_20",
        f"{var}_20_plus"
    ]

    # Create bin column
    Weather[f"{var}_bin"] = pd.cut(
        Weather[var],
        bins=rain_bins,
        labels=rain_labels,
        right=False
    )

    # Create binary columns for each bin
    for label in rain_labels:
        Weather[label] = (Weather[f"{var}_bin"] == label).astype(int)

    # Drop temporary bin column
    Weather.drop(columns=f"{var}_bin", inplace=True)

'''
#Defining Relative Temperature & SD Bins
#Weather['idw_tavg_fahrenheit'] = (Weather['idw_tavg'] * 1.8) + 32
#Weather['tavg_norm_monavg_fahrenheit'] = (Weather['idw_dlytavg_norm_monavg'] * 1.8) + 32
#Weather['tavg_norm'] = (Weather['nn_tavg'] - Weather['nn_mean_tavg'])/Weather['nn_sd_tavg']

#var = "tavg_norm"   # already normalised (z-score)
#zcol = var

# 1) Define range for middle bins (-2 SD to +2 SD, 0.5 SD increments)
#middle_bins = np.arange(-3, 3, 0.5)

# 2) Create 0.5 SD bins for -2 ≤ z < +2
for num in middle_bins:
    col_name = f"{zcol}_SD_{num:.1f}".replace("-", "m")
    Weather[col_name] = (
        (Weather[zcol] >= num) & (Weather[zcol] < num + 0.5)
    ).astype(int)

# 3) Lowest bin: below -2 SD
Weather[f"{zcol}_below_m3SD"] = (Weather[zcol] < -3).astype(int)

# 4) Highest bin: above +2 SD
Weather[f"{zcol}_above_3SD"] = (Weather[zcol] >= 3).astype(int)
'''

# Checked if code is correct 24/09/2024 
clear_env(keep=["directory", "Weather"])


C:\Users\20234503\AppData\Local\Temp\ipykernel_14748\4017174325.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Weather[fahr_col] = (Weather[var] * 1.8) + 32
C:\Users\20234503\AppData\Local\Temp\ipykernel_14748\4017174325.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Weather[col_name] = (np.floor(Weather[fahr_col] * 2) / 2 == num).astype(int)
C:\Users\20234503\AppData\Local\Temp\ipykernel_14748\4017174325.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a Data

In [7]:
#######################################
# Creation of Lagged Temperature Bins # (Run Time: 31:17)
#######################################
weather_vars = [
    # tavg 5°F bins
    'idw_tavg_below_10',
    'idw_tavg_10_15', 'idw_tavg_15_20', 'idw_tavg_20_25',
    'idw_tavg_25_30', 'idw_tavg_30_35', 'idw_tavg_35_40',
    'idw_tavg_40_45', 'idw_tavg_45_50', 'idw_tavg_50_55',
    'idw_tavg_55_60', 'idw_tavg_60_65', 'idw_tavg_65_70',
    'idw_tavg_70_75', 'idw_tavg_75_80', 'idw_tavg_80_85',
    'idw_tavg_85_90',
    'idw_tavg_above_90',

    # tavg 5°F bins
    'nn_tavg_below_10',
    'nn_tavg_10_15', 'nn_tavg_15_20', 'nn_tavg_20_25',
    'nn_tavg_25_30', 'nn_tavg_30_35', 'nn_tavg_35_40',
    'nn_tavg_40_45', 'nn_tavg_45_50', 'nn_tavg_50_55',
    'nn_tavg_55_60', 'nn_tavg_60_65', 'nn_tavg_65_70',
    'nn_tavg_70_75', 'nn_tavg_75_80', 'nn_tavg_80_85',
    'nn_tavg_85_90',
    'nn_tavg_above_90',


    # tmin 5°F bins
    #'idw_tmin_below_10',
    #'idw_tmin_10.0_15.0', 'idw_tmin_15.0_20.0', 'idw_tmin_20.0_25.0',
    #'idw_tmin_25.0_30.0', 'idw_tmin_30.0_35.0', 'idw_tmin_35.0_40.0',
    #'idw_tmin_40.0_45.0', 'idw_tmin_45.0_50.0', 'idw_tmin_50.0_55.0',
    #'idw_tmin_55.0_60.0', 'idw_tmin_60.0_65.0', 'idw_tmin_65.0_70.0',
    #'idw_tmin_70.0_75.0', 'idw_tmin_75.0_80.0', 'idw_tmin_80.0_85.0',
    #'idw_tmin_85.0_90.0',
    #'idw_tmin_above_90',

    # tmax 5°F bins
    #'idw_tmax_below_10',
    #'idw_tmax_10.0_15.0', 'idw_tmax_15.0_20.0', 'idw_tmax_20.0_25.0',
    #'idw_tmax_25.0_30.0', 'idw_tmax_30.0_35.0', 'idw_tmax_35.0_40.0',
    #'idw_tmax_40.0_45.0', 'idw_tmax_45.0_50.0', 'idw_tmax_50.0_55.0',
    #'idw_tmax_55.0_60.0', 'idw_tmax_60.0_65.0', 'idw_tmax_65.0_70.0',
    #'idw_tmax_70.0_75.0', 'idw_tmax_75.0_80.0', 'idw_tmax_80.0_85.0',
    #'idw_tmax_85.0_90.0',
    #'idw_tmax_above_90',

    # precip bins
    'idw_prcp_0_1', 'idw_prcp_1_10', 'idw_prcp_10_20', 'idw_prcp_20_plus',
    'nn_prcp_0_1', 'nn_prcp_1_10', 'nn_prcp_10_20', 'nn_prcp_20_plus',
    
    # normalised tavg bins (0.5 SD bins with tails)
    #'tavg_norm_below_m3SD', 'tavg_norm_SD_m3.0', 'tavg_norm_SD_m2.5', 'tavg_norm_SD_m2.0',
    #'tavg_norm_SD_m1.5', 'tavg_norm_SD_m1.0',
    #'tavg_norm_SD_m0.5', 'tavg_norm_SD_0.0',
    #'tavg_norm_SD_0.5', 'tavg_norm_SD_1.0',
    #'tavg_norm_SD_1.5', 'tavg_norm_SD_2.0', 'tavg_norm_SD_2.5',
    #'tavg_norm_above_3SD',

]


# Define lag periods (in days)
lag_windows = [7, 14, 30, 60, 90]
#lag_windows = [30]
# Generate lagged and lead variables
for var in weather_vars:
    for days in lag_windows:
        # Lag
        lagged_name = f"{var}_lag_{days}d"
        Weather[lagged_name] = (
            Weather.groupby('FIPS')[var]
            .transform(lambda x: x.shift(1).rolling(window=days, min_periods=1).sum())
        )

        # Lead
        lead_name = f"{var}_lead_{days}d"
        Weather[lead_name] = (
            Weather.groupby('FIPS')[var]
            .transform(lambda x: x.shift(-days).rolling(window=days, min_periods=1).sum())
        )

# === Save the final DataFrame === #
Weather.to_csv(directory+"Data\\weather.csv", index=False)

C:\Users\20234503\AppData\Local\Temp\ipykernel_14748\1192081640.py:69: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Weather[lagged_name] = (
C:\Users\20234503\AppData\Local\Temp\ipykernel_14748\1192081640.py:76: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Weather[lead_name] = (
C:\Users\20234503\AppData\Local\Temp\ipykernel_14748\1192081640.py:69: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the ca

In [6]:
# === Save the final DataFrame === #
Weather.to_csv(r"C:\Users\20234503\Dropbox\PhD Project\Research\Weather and Strikes\weather.csv", index=False)

KeyboardInterrupt: 

In [ ]:
C:\Users\20234503\Dropbox\PhD Project\Research\Weather and Strikes